## Download and process ONC AZFP data

This notebook demonstrates how to request, download, calibrate, and grid raw Acoustic Zooplankton and Fish Profiler (AZFP) data from Ocean Networks Canada (ONC) using the ONC Python API and Echopype.

An ONC account and API token are required to download the data. Your token is available from your [ONC account profile](https://data.oceannetworks.ca/DataSearch). Interested users can find additional examples in the [ONC echosounder examples repository](https://github.com/slonimer/onc-echosounder-examples)!

### Import packages and configure the download

In [1]:
from pathlib import Path
import echopype as ep
from onc import ONC

download_dir = Path("example_data/ONC").resolve()
download_dir.mkdir(parents=True, exist_ok=True)

onc = ONC(
    token="a2b37084-c22c-4c6a-a1fd-f8f7fec2af2b",
    outPath=download_dir,
)

### Download the data

Define the data-product request filters. Here, we request one hour of raw data from AZFP device `ASLAZFP55196`. The request covers 7 July 2024 from 00:00 to 01:00 UTC.

In [2]:
filters = {
    "deviceCode": "ASLAZFP55196",
    "dataProductCode": "AAPTS",
    "extension": "01a",
    "dateFrom": "2024-07-07T00:00:00.000Z",
    "dateTo": "2024-07-07T01:00:00.000Z",
    "dpo_aslBinarySource": 0,
}

request = onc.requestDataProduct(filters)

print("Request ID:", request["dpRequestId"])
print("Estimated size:", request["estimatedFileSize"])

Request Id: 46287610
Estimated File Size: 17 MB
Estimated Processing Time: 10 s
Request ID: 46287610
Estimated size: 17 MB


Submit the request and wait for ONC to generate the data-product files.

In [3]:
run = onc.runDataProduct(
    request["dpRequestId"],
    waitComplete=True,
)

print("Files generated:", run["fileCount"])

To cancel the running data product, run 'onc.cancelDataProduct(46287610)'

   queued
   data product running.........
   3 files generated for this data product
   complete
Files generated: 3


Download the generated files to the local data directory.

In [ ]:
downloaded_files = onc.downloadDataProduct(run["runIds"][0])



   Search complete, waiting on the file system to synchronize (ASLAZFP55196_20240707T000000.578Z_20240707T005955.539Z.01a)......

### Open the AZFP file

Identify the downloaded AZFP binary file and its corresponding instrument configuration file. The ONC metadata XML file is excluded because Echopype requires the AZFP instrument configuration XML.

In [ ]:
raw_files = list(download_dir.glob("*.01a"))
xml_files = [
    path
    for path in download_dir.glob("*.xml")
    if "_META" not in path.name.upper()
]

if len(raw_files) != 1:
    raise ValueError(f"Expected one .01a file, found {len(raw_files)}.")

if len(xml_files) != 1:
    raise ValueError(
        f"Expected one AZFP configuration XML file, found {len(xml_files)}."
    )

raw_file = raw_files[0]
xml_file = xml_files[0]

print("Raw file:", raw_file.name)
print("Configuration file:", xml_file.name)

Open the raw AZFP file using its associated instrument configuration XML.

In [ ]:
echodata = ep.open_raw(
    raw_file=raw_file,
    sonar_model="AZFP",
    xml_path=xml_file,
)

echodata

### Inspect instrument and environmental metadata

The nominal frequency identifies the frequency associated with each acoustic channel. AZFP instruments also record internal temperature, whereas salinity and deployment pressure generally need to be obtained from accompanying deployment or environmental metadata.

In [ ]:
frequency_nominal = echodata["Sonar/Beam_group1"]["frequency_nominal"]
frequency_nominal

In [ ]:
echodata["Environment"]

In [ ]:
echodata["Platform"]

### Calibrate volume backscattering strength

For this example, salinity and pressure are supplied explicitly, while Echopype uses the temperature stored in the AZFP data. Here, pressure is approximated from a transducer depth of 165 m.

In [ ]:
transducer_depth = 165.0  # m

env_params = {
    "salinity": 32,               # PSU; to verify
    "pressure": transducer_depth, # dbar; approximate value at 165 m depth
}

ds_Sv = ep.calibrate.compute_Sv(
    echodata,
    env_params=env_params,
).set_coords("echo_range")

ds_Sv

Inspect the environmental parameters and derived quantities used during calibration.

In [ ]:
calibration_environment = [
    name
    for name in [
        "temperature",
        "salinity",
        "pressure",
        "sound_speed",
        "sound_absorption",
    ]
    if name in ds_Sv
]

ds_Sv[calibration_environment]

### Visualize calibrated data by range

`echo_range` represents distance from the transducer along the acoustic beam. At this stage, the vertical axis is relative to the transducer rather than to the sea surface.

In [ ]:
range_plot = ds_Sv["Sv"].plot.pcolormesh(
    x="ping_time",
    y="echo_range",
    col="channel",
    col_wrap=1,
    yincrease=True,
    cmap="viridis",
    vmin=-100,
    vmax=-40,
    size=3,
    aspect=3,
)

range_plot.set_xlabels("Time")
range_plot.set_ylabels("Range from transducer (m)")

### Convert range to depth below the sea surface

This AZFP is mounted on the seafloor and points upward. The `depth_offset` below is the depth of the transducer itself.

In [ ]:
ds_Sv_depth = ep.consolidate.add_depth(
    ds_Sv,
    depth_offset=transducer_depth,
    downward=False,
).set_coords("depth")

ds_Sv_depth["depth"]

Before gridding, `depth` can vary across channel, time, and range sample, so it is a multidimensional auxiliary coordinate rather than a one-dimensional dimension. Explicit `pcolormesh` plotting allows xarray to use this physical coordinate.

In [ ]:
depth_plot = ds_Sv_depth["Sv"].plot.pcolormesh(
    x="ping_time",
    y="depth",
    col="channel",
    col_wrap=1,
    yincrease=False,
    cmap="viridis",
    vmin=-100,
    vmax=-40,
    size=3,
    aspect=3,
)

depth_plot.set_xlabels("Time")
depth_plot.set_ylabels("Depth below sea surface (m)")

### Compute MVBS on a depth grid

Compute mean volume backscattering strength (MVBS) using 0.5 m absolute-depth bins and 30 s time bins. A distinct object name emphasizes that this product is gridded by depth, not by range from the transducer.

In [ ]:
ds_MVBS_depth = ep.commongrid.compute_MVBS(
    ds_Sv_depth,
    range_var="depth",
    range_bin="0.5m",
    ping_time_bin="30s",
)

ds_MVBS_depth

Verify the dimensions before plotting the depth-gridded MVBS product.

In [ ]:
mvbs_plot = ds_MVBS_depth["Sv"].plot.pcolormesh(
    x="ping_time",
    y="depth",
    col="channel",
    col_wrap=1,
    yincrease=False,
    cmap="viridis",
    vmin=-100,
    vmax=-40,
    size=3,
    aspect=3,
)

mvbs_plot.set_xlabels("Time")
mvbs_plot.set_ylabels("Depth below sea surface (m)")

### Verify channel and frequency mapping

Compare the channel-to-frequency mapping before and after gridding. Each channel must remain associated with its original nominal frequency.

In [ ]:
print("Input Sv:")
for channel, frequency in zip(
    ds_Sv_depth["channel"].values,
    ds_Sv_depth["frequency_nominal"].values,
):
    print(channel, frequency)

print("\nDepth-gridded MVBS:")
for channel, frequency in zip(
    ds_MVBS_depth["channel"].values,
    ds_MVBS_depth["frequency_nominal"].values,
):
    print(channel, frequency)

End of notebook

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version

print(f"Executed: {datetime.now(timezone.utc):%Y-%m-%d %H:%M:%S UTC}")
print(f"echopype: {ep.__version__}")

for package in ["xarray", "numpy", "matplotlib", "onc"]:
    print(f"{package}: {version(package)}")